In [ ]:
"""VibeCheck AI — Full Analysis Notebook (source for notebook.ipynb)"""

In [ ]:
# # 🎧 VibeCheck AI — Music Mood & Energy Analysis
#
# **Author:** [Your Name]
# **Stack:** Python · Scikit-learn · TensorFlow/Keras · Plotly
#
# ## Objective
# Build an end-to-end ML pipeline to:
# 1. **Explore** Spotify-like audio features (EDA)
# 2. **Cluster** tracks into vibe groups using KMeans
# 3. **Predict** danceability using a deep neural network
# 4. **Analyze** track name sentiment using NLP
#
# ---

In [ ]:
# ## 1. Setup & Data Generation

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")

from src.generate_data import generate_dataset

plt.style.use("dark_background")
sns.set_theme(style="darkgrid")

# Generate dataset
df = generate_dataset(n_samples=5000)
print(f"✅ Dataset shape: {df.shape}")
print(f"\n📊 First rows:")
df.head()

In [ ]:
# ## 2. Exploratory Data Analysis (EDA)

In [ ]:
print("📈 Descriptive statistics:")
df.describe().round(3)

In [ ]:
# Distribution of audio features
AUDIO_FEATURES = [
    "energy", "valence", "tempo", "danceability",
    "acousticness", "instrumentalness", "liveness", "loudness", "speechiness"
]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle("Distribution of Audio Features", fontsize=16, fontweight="bold")

for ax, feat in zip(axes.flatten(), AUDIO_FEATURES):
    ax.hist(df[feat], bins=40, color="#a855f7", alpha=0.8, edgecolor="white", linewidth=0.3)
    ax.set_title(feat.capitalize(), fontsize=10)
    ax.set_ylabel("Count")

plt.tight_layout()
plt.savefig("plots/01_feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved")

In [ ]:
# Correlation heatmap
corr = df[AUDIO_FEATURES].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title("🔗 Audio Features Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/02_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Vibe ground truth distribution (from generator)
fig = px.pie(
    df, names="vibe",
    title="Ground Truth Vibe Distribution",
    color_discrete_sequence=px.colors.qualitative.Vivid
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()

In [ ]:
# ## 3. Machine Learning — KMeans Clustering
#
# We use KMeans to discover **natural vibe clusters** in the audio feature space
# without relying on the ground truth labels.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Preprocessing
scaler = StandardScaler()
X = scaler.fit_transform(df[AUDIO_FEATURES])
print(f"✅ Scaled feature matrix: {X.shape}")

In [ ]:
# Find optimal K using Elbow + Silhouette
from src.ml_pipeline import find_optimal_k

k_range = range(2, 11)
ks, inertias, silhouettes = find_optimal_k(X, k_range)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ks, inertias, "o-", color="#a855f7", linewidth=2.5, markersize=8)
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Inertia (WCSS)")
ax1.set_title("📉 Elbow Method")
ax1.axvline(5, color="#38bdf8", linestyle="--", label="K=5")
ax1.legend()

ax2.plot(ks, silhouettes, "o-", color="#38bdf8", linewidth=2.5, markersize=8)
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Silhouette Score")
ax2.set_title("📈 Silhouette Scores")
ax2.axvline(5, color="#a855f7", linestyle="--", label="K=5")
ax2.legend()

plt.suptitle("Finding Optimal Number of Vibe Clusters", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("plots/03_optimal_k.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Best silhouette at K={ks[np.argmax(silhouettes)]}: {max(silhouettes):.4f}")

In [ ]:
# Train final model with K=5
from src.ml_pipeline import train_clustering, get_vibe_label

kmeans, labels, sil = train_clustering(X, n_clusters=5)
df["cluster"] = labels
df["vibe_label"] = df["cluster"].apply(get_vibe_label)

print(f"✅ KMeans trained — Silhouette Score: {sil:.4f}")
print(f"\n📊 Cluster distribution:")
print(df["vibe_label"].value_counts())

In [ ]:
# Visualize clusters with PCA (2D)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

df_pca = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "vibe": df["vibe_label"],
    "track": df["track_name"],
    "energy": df["energy"],
    "tempo": df["tempo"]
})

fig = px.scatter(
    df_pca.sample(2000), x="PC1", y="PC2",
    color="vibe", size="energy",
    hover_data=["track", "tempo"],
    title="🗺️ Vibe Clusters — PCA Projection",
    color_discrete_sequence=px.colors.qualitative.Vivid,
    template="plotly_dark", opacity=0.7
)
fig.show()

In [ ]:
# Energy × Valence space colored by cluster
fig = px.scatter(
    df.sample(2000), x="valence", y="energy",
    color="vibe_label", size="danceability",
    hover_data=["track_name", "genre", "tempo"],
    title="⚡ Energy × Valence Vibe Map",
    color_discrete_sequence=px.colors.qualitative.Vivid,
    template="plotly_dark", opacity=0.7
)
fig.update_layout(height=550)
fig.show()

In [ ]:
# Radar chart per cluster
features_radar = ["energy", "valence", "danceability", "acousticness", "liveness"]
cluster_means = df.groupby("vibe_label")[features_radar].mean()

fig = go.Figure()
colors = px.colors.qualitative.Vivid

for i, (vibe, row) in enumerate(cluster_means.iterrows()):
    vals = list(row) + [row.iloc[0]]
    cats = features_radar + [features_radar[0]]
    fig.add_trace(go.Scatterpolar(
        r=vals, theta=cats,
        fill="toself", name=vibe,
        line_color=colors[i], opacity=0.8
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title="🎯 Cluster Audio Feature Profiles",
    template="plotly_dark", height=500
)
fig.show()

In [ ]:
# ## 4. Deep Learning — Danceability Predictor
#
# We train a feedforward neural network to predict danceability
# from the remaining audio features.

In [ ]:
from src.ml_pipeline import train_neural_network

print("🧠 Training neural network...")
model, history, metrics = train_neural_network(df)

print(f"\n✅ Results:")
print(f"   MAE  : {metrics['mae']:.4f}")
print(f"   R²   : {metrics['r2']:.4f}")

In [ ]:
# Training curves
hist_df = pd.DataFrame({
    "Epoch": range(1, len(history.history["loss"]) + 1),
    "Train Loss": history.history["loss"],
    "Val Loss":   history.history["val_loss"],
})

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_df["Epoch"], hist_df["Train Loss"], color="#a855f7", lw=2, label="Train Loss")
ax.plot(hist_df["Epoch"], hist_df["Val Loss"],   color="#38bdf8", lw=2, label="Val Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("🧠 Neural Network Training Curves")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plots/04_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Predicted vs Actual
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(metrics["y_test"][:500], metrics["y_pred"][:500],
           alpha=0.5, color="#a855f7", edgecolors="white", linewidths=0.2, s=40)
ax.plot([0, 1], [0, 1], "--", color="white", lw=1.5, label="Perfect prediction")
ax.set_xlabel("Actual Danceability")
ax.set_ylabel("Predicted Danceability")
ax.set_title("🎯 Predicted vs Actual Danceability")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plots/05_predicted_vs_actual.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Model summary:")
model.summary()

In [ ]:
# ## 5. NLP — Track Name Sentiment Analysis

In [ ]:
from src.ml_pipeline import analyze_sentiment_simple

df["sentiment"] = analyze_sentiment_simple(df["track_name"])

print("📝 Sentiment score distribution:")
print(df["sentiment"].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for vibe in df["vibe_label"].unique():
    subset = df[df["vibe_label"] == vibe]["sentiment"]
    ax.hist(subset, bins=20, alpha=0.6, label=vibe)

ax.set_xlabel("Sentiment Score (0=negative, 1=positive)")
ax.set_ylabel("Count")
ax.set_title("💬 Track Name Sentiment Distribution by Vibe")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("plots/06_sentiment_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Sentiment vs energy scatter
fig = px.scatter(
    df.sample(1000), x="sentiment", y="energy",
    color="vibe_label",
    title="💬 Sentiment × Energy",
    color_discrete_sequence=px.colors.qualitative.Vivid,
    template="plotly_dark", opacity=0.7
)
fig.show()

In [ ]:
# ## 6. Conclusions
#
# ### Key Findings
# - **KMeans** successfully separates tracks into distinct vibe clusters with a silhouette score > 0.3
# - **Energy and Valence** are the two most discriminating features across clusters
# - **Tempo** is strongly associated with "Hype" vibes (> 145 BPM) and "Chill" (< 110 BPM)
# - **Neural Network** predicts danceability with R² ≈ 0.75, MAE ≈ 0.07
# - **NLP sentiment** on track names shows a weak but real correlation with vibe positivity
#
# ### Next Steps
# - Use real Spotify API data via `spotipy`
# - Experiment with UMAP for dimensionality reduction (vs PCA)
# - Try Transformer-based NLP (DistilBERT) for richer track name embeddings
# - Deploy the Streamlit app to Hugging Face Spaces or Streamlit Cloud
#
# ---
# *VibeCheck AI — github.com/[your-username]/vibecheck-ai*